# 02 — Наша статическая модель прогнозирования

## Цель

На этом этапе строится **наша собственная базовая прогнозная архитектура** для муниципальных расходов.

Она состоит из двух разных экспертов:

1. **Panel AR** — прогнозирует общий рост панели и локальное отклонение конкретного МО;
2. **Seasonal expert** — использует устойчивую сезонную форму, региональный профиль и поведенческий кластер.

Вес двух экспертов выбирается **только на более раннем периоде**, до основной оценки.

## Принцип честной проверки

- калибровка весов: ретроспективные целевые месяцы вплоть до **июня 2024 включительно** (`target_period <= 2024-06-01`);
- основная проверка: origins `2024-06 ... 2024-09`, горизонты `1–3`;
- дополнительный forward-период: origins `2024-10` и `2024-11`;
- будущие фактические значения не используются при построении прогноза.

**Оговорка о доступности:** `period` обозначает отчётный месяц, не фактическую дату публикации статистики. Поэтому ретроспективная калибровка на июне не доказывает, что веса 45/55 были доступны в реальном времени на 1 июня. Здесь веса сохраняются как зафиксированная конфигурация исторического benchmark; исходные вычисления не переоптимизируются.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

# Разрешаем запуск как из корня проекта, так и из каталога notebooks/.
# Предполагается стандартная структура проекта: data/interim и notebooks/.
CURRENT_DIR = Path.cwd().resolve()
if (CURRENT_DIR / "data" / "interim").is_dir():
    PROJECT_DIR = CURRENT_DIR
elif (CURRENT_DIR.parent / "data" / "interim").is_dir():
    PROJECT_DIR = CURRENT_DIR.parent
else:
    raise FileNotFoundError(
        "Не найден data/interim. Запускайте ноутбук из корня проекта "
        "или из каталога notebooks/."
    )
INTERIM_DIR = PROJECT_DIR / "data" / "interim"

PANEL_FILE = INTERIM_DIR / "consumption_total_panel_2016.parquet"
GEO_FILE = INTERIM_DIR / "consumption_series_geography.parquet"
CLUSTER_FILE = INTERIM_DIR / "seasonal_cluster_map_k5.parquet"

required = {
    "panel": PANEL_FILE,
    "geography": GEO_FILE,
    "cluster_map": CLUSTER_FILE,
}

check = pd.DataFrame([
    {"artifact": name, "path": str(path), "exists": path.exists()}
    for name, path in required.items()
])

display(check)

missing = check.loc[~check["exists"], "artifact"].tolist()
if missing:
    raise FileNotFoundError(
        "Не найдены необходимые служебные артефакты: "
        + ", ".join(missing)
    )

print("Входные данные: OK")


## 1. Загрузка данных

Основная панель содержит 2 016 полных муниципальных рядов за 24 месяца.

Дополнительно используются заранее подготовленные служебные метаданные:

- региональная принадлежность;
- поведенческий сезонный кластер.

Они нужны только сезонному эксперту.

In [ ]:
panel = pd.read_parquet(PANEL_FILE)
geography = pd.read_parquet(GEO_FILE)
cluster_map = pd.read_parquet(CLUSTER_FILE)

panel["period"] = pd.to_datetime(panel["period"])

print("Панель:", panel.shape)
print("Рядов:", panel["series_id"].nunique())
print("Период:", panel["period"].min(), "—", panel["period"].max())
print("География:", geography.shape)
print("Кластеры:", cluster_map.shape)

assert panel.shape == (48_384, 4)
assert panel["series_id"].nunique() == 2016
assert panel["period"].nunique() == 24
assert not panel.duplicated(["series_id", "period"]).any()
assert panel.isna().sum().sum() == 0

print("Проверка панели: PASSED")


## 2. Эксперт Panel AR

Идея простая:

1. для каждого МО считаем годовой логарифмический рост;
2. медиана по всем МО образует **общий фактор панели**;
3. разница между ростом МО и общим фактором — локальное отклонение;
4. локальное отклонение прогнозируется простой pooled AR(1)-моделью;
5. итоговый прогноз восстанавливается относительно значения того же месяца прошлого года.

Так модель отдельно учитывает **общую динамику рынка** и **индивидуальное поведение МО**.

In [ ]:
# Подготовка состояния Panel AR
panel_ar_data = (
    panel
    .sort_values(["series_id", "period"])
    .copy()
)

panel_ar_data["lag_12"] = (
    panel_ar_data
    .groupby("series_id")["value"]
    .shift(12)
)

panel_ar_data["log_yoy"] = np.log(
    panel_ar_data["value"] / panel_ar_data["lag_12"]
)

panel_factor = (
    panel_ar_data
    .dropna(subset=["log_yoy"])
    .groupby("period")["log_yoy"]
    .median()
    .sort_index()
)

panel_ar_data["panel_factor"] = panel_ar_data["period"].map(panel_factor)

panel_ar_data["panel_residual"] = (
    panel_ar_data["log_yoy"]
    - panel_ar_data["panel_factor"]
)

panel_ar_data["residual_lag1"] = (
    panel_ar_data
    .groupby("series_id")["panel_residual"]
    .shift(1)
)

print("Доступных значений общего фактора:", len(panel_factor))
print("Период фактора:", panel_factor.index.min(), "—", panel_factor.index.max())


In [ ]:
def forecast_panel_factor(
    origin,
    horizons=(1, 2, 3),
    last_weight=0.65,
    trend_weight=0.35,
):
    # Рекурсивный прогноз общего фактора панели.
    origin = pd.Timestamp(origin)
    horizons = sorted(int(h) for h in horizons)
    max_h = max(horizons)

    history = (
        panel_factor.loc[panel_factor.index <= origin]
        .astype(float)
        .tolist()
    )

    if len(history) < 3:
        raise ValueError("Для прогноза фактора нужно минимум 3 наблюдения.")

    rows = []

    for h in range(1, max_h + 1):
        last_factor = history[-1]

        y3 = np.asarray(history[-3:], dtype=float)
        x3 = np.arange(3, dtype=float)

        slope, intercept = np.polyfit(x3, y3, 1)
        trend_next = intercept + slope * 3

        next_factor = (
            last_weight * last_factor
            + trend_weight * trend_next
        )

        history.append(next_factor)

        if h in horizons:
            rows.append({
                "horizon": h,
                "target_period": origin + pd.DateOffset(months=h),
                "factor_pred": next_factor,
            })

    return pd.DataFrame(rows)


def forecast_panel_ar(origin, horizons=(1, 2, 3)):
    # Каузальный pooled AR(1) прогноз для всех 2 016 рядов.
    origin = pd.Timestamp(origin)
    horizons = sorted(int(h) for h in horizons)

    train = panel_ar_data[
        (panel_ar_data["period"] <= origin)
        & panel_ar_data["panel_residual"].notna()
        & panel_ar_data["residual_lag1"].notna()
    ]

    x = train["residual_lag1"].to_numpy()
    y = train["panel_residual"].to_numpy()

    beta = np.sum(x * y) / np.sum(x ** 2)

    origin_residual = (
        panel_ar_data[
            panel_ar_data["period"] == origin
        ][["series_id", "panel_residual"]]
        .rename(columns={"panel_residual": "origin_residual"})
        .copy()
    )

    if origin_residual.empty:
        raise ValueError(f"Нет данных для origin={origin}")

    factor_fc = forecast_panel_factor(
        origin=origin,
        horizons=horizons,
    )

    parts = []

    for h in horizons:
        target_period = origin + pd.DateOffset(months=h)

        factor_pred = float(
            factor_fc.loc[
                factor_fc["horizon"] == h,
                "factor_pred",
            ].iloc[0]
        )

        seasonal_period = target_period - pd.DateOffset(months=12)

        seasonal_base = (
            panel[
                panel["period"] == seasonal_period
            ][["series_id", "value"]]
            .rename(columns={"value": "seasonal_base"})
        )

        part = (
            origin_residual
            .merge(
                seasonal_base,
                on="series_id",
                how="left",
                validate="one_to_one",
            )
        )

        part["origin"] = origin
        part["horizon"] = h
        part["target_period"] = target_period
        part["beta"] = beta
        part["factor_pred"] = factor_pred

        part["residual_pred"] = (
            (beta ** h) * part["origin_residual"]
        )

        part["panel_ar_pred"] = (
            part["seasonal_base"]
            * np.exp(
                part["factor_pred"]
                + part["residual_pred"]
            )
        )

        parts.append(part)

    result = pd.concat(parts, ignore_index=True)

    assert len(result) == 2016 * len(horizons)
    assert result["panel_ar_pred"].notna().all()

    return result


## 3. Сезонный эксперт

Сезонный эксперт использует только информацию, доступную заранее:

- сезонные профили строятся по истории **2023 года**;
- региональный профиль описывает типичное изменение в регионе;
- поведенческий кластер объединяет МО с похожей сезонной формой;
- если регион или кластер недоступен, используется более общий профиль.

Итоговый месячный рост — среднее регионального и кластерного профилей.

In [ ]:
# Служебные метаданные сезонного эксперта
frozen_meta = (
    geography[
        ["series_id", "region_code", "match_method"]
    ]
    .copy()
)

frozen_meta["frozen_region_code"] = np.where(
    frozen_meta["match_method"].eq("deterministic"),
    frozen_meta["region_code"].astype(str),
    "UNK",
)

frozen_meta = (
    frozen_meta[
        ["series_id", "frozen_region_code"]
    ]
    .merge(
        cluster_map[["series_id", "cluster_id"]],
        on="series_id",
        how="left",
        validate="one_to_one",
    )
)

# Сезонные профили оцениваем только по 2023 году
frozen_series = set(cluster_map["series_id"])

seasonal_hist = (
    panel[
        panel["series_id"].isin(frozen_series)
    ][["series_id", "period", "value"]]
    .merge(
        frozen_meta,
        on="series_id",
        how="left",
        validate="many_to_one",
    )
    .sort_values(["series_id", "period"])
    .copy()
)

seasonal_hist["lag_1"] = (
    seasonal_hist
    .groupby("series_id")["value"]
    .shift(1)
)

seasonal_hist["monthly_growth"] = (
    seasonal_hist["value"]
    / seasonal_hist["lag_1"]
    - 1
)

seasonal_hist["month"] = seasonal_hist["period"].dt.month

hist_2023 = seasonal_hist[
    seasonal_hist["period"].dt.year == 2023
].copy()

national_profile = (
    hist_2023
    .groupby("month")["monthly_growth"]
    .median()
    .rename("national_growth")
    .reset_index()
)

region_profile = (
    hist_2023[
        hist_2023["frozen_region_code"] != "UNK"
    ]
    .groupby(["frozen_region_code", "month"])["monthly_growth"]
    .median()
    .rename("region_growth")
    .reset_index()
)

cluster_profile = (
    hist_2023
    .groupby(["cluster_id", "month"])["monthly_growth"]
    .median()
    .rename("cluster_growth")
    .reset_index()
)

print("Рядов с кластером:", frozen_meta["cluster_id"].notna().sum())
print("Месяцев в национальном профиле:", len(national_profile))


In [ ]:
def forecast_seasonal(origin, horizons=(1, 2, 3)):
    # Рекурсивный сезонный прогноз для всех 2 016 рядов.
    origin = pd.Timestamp(origin)
    horizons = sorted(int(h) for h in horizons)
    max_h = max(horizons)

    state = (
        panel[
            panel["period"] == origin
        ][["series_id", "value"]]
        .rename(columns={"value": "recursive_value"})
        .merge(
            frozen_meta,
            on="series_id",
            how="left",
            validate="one_to_one",
        )
    )

    if state.empty:
        raise ValueError(f"Нет данных для origin={origin}")

    parts = []

    for h in range(1, max_h + 1):
        target_period = origin + pd.DateOffset(months=h)
        month = target_period.month

        national_values = (
            national_profile.loc[
                national_profile["month"] == month,
                "national_growth",
            ]
        )

        if national_values.empty:
            raise ValueError(f"Нет сезонного профиля для месяца {month}")

        national_g = float(national_values.iloc[0])

        if pd.isna(national_g):
            raise ValueError(
                f"Сезонный рост для месяца {month} не определён."
            )

        step = state.copy()
        step["month"] = month
        step["national_growth"] = national_g

        step = step.merge(
            region_profile,
            on=["frozen_region_code", "month"],
            how="left",
            validate="many_to_one",
        )

        step["region_growth"] = (
            step["region_growth"]
            .fillna(step["national_growth"])
        )

        step = step.merge(
            cluster_profile,
            on=["cluster_id", "month"],
            how="left",
            validate="many_to_one",
        )

        step["cluster_growth"] = (
            step["cluster_growth"]
            .fillna(step["region_growth"])
        )

        step["seasonal_growth"] = (
            0.5 * step["region_growth"]
            + 0.5 * step["cluster_growth"]
        )

        step["seasonal_pred"] = (
            step["recursive_value"]
            * (1.0 + step["seasonal_growth"])
        )

        if h in horizons:
            out = step[
                ["series_id", "seasonal_pred"]
            ].copy()

            out["origin"] = origin
            out["horizon"] = h
            out["target_period"] = target_period
            parts.append(out)

        state = (
            step[
                [
                    "series_id",
                    "frozen_region_code",
                    "cluster_id",
                    "seasonal_pred",
                ]
            ]
            .rename(columns={"seasonal_pred": "recursive_value"})
        )

    result = pd.concat(parts, ignore_index=True)

    assert len(result) == 2016 * len(horizons)
    assert result["seasonal_pred"].notna().all()

    return result


## 4. Ретроспективная калибровка веса

Вес двух экспертов выбирается **до основной тестовой части**.

Используются прогнозы, целевые месяцы которых не позже **июня 2024 включительно** (`target_period <= 2024-06-01`):

- март: `h1`, `h2`, `h3`;
- апрель: `h1`, `h2`;
- май: `h1`.

Всего: `12 096` строк.

Проверяем простую сетку весов с шагом `0.05`.

Это хронологическое разделение целевых месяцев, а не гарантия публикационной доступности данных на дату июньского origin. Для строгого сценария real-time требовалась бы отдельная проверка задержки публикации и перерасчёт калибровки, не входящий в финальный cleanup.

In [ ]:
CAL_ORIGINS = pd.to_datetime([
    "2024-03-01",
    "2024-04-01",
    "2024-05-01",
])

CAL_CUTOFF = pd.Timestamp("2024-06-01")

cal_parts = []

for origin in CAL_ORIGINS:
    available_h = [
        h for h in [1, 2, 3]
        if origin + pd.DateOffset(months=h) <= CAL_CUTOFF
    ]

    panel_fc = forecast_panel_ar(origin, available_h)[
        ["series_id", "origin", "horizon", "target_period", "panel_ar_pred"]
    ]

    seasonal_fc = forecast_seasonal(origin, available_h)[
        ["series_id", "origin", "horizon", "target_period", "seasonal_pred"]
    ]

    actual = (
        panel[
            ["series_id", "period", "value"]
        ]
        .rename(
            columns={
                "period": "target_period",
                "value": "actual_value",
            }
        )
    )

    part = (
        panel_fc
        .merge(
            seasonal_fc,
            on=["series_id", "origin", "horizon", "target_period"],
            validate="one_to_one",
        )
        .merge(
            actual,
            on=["series_id", "target_period"],
            validate="many_to_one",
        )
    )

    cal_parts.append(part)

calibration = pd.concat(cal_parts, ignore_index=True)

print("Калибровочных строк:", len(calibration))

display(
    calibration[
        ["origin", "horizon", "target_period"]
    ]
    .drop_duplicates()
    .sort_values(["origin", "horizon"])
)

assert len(calibration) == 12_096
assert calibration["target_period"].max() <= CAL_CUTOFF


In [ ]:
weight_rows = []

for panel_weight in np.round(np.arange(0.0, 1.0001, 0.05), 2):
    seasonal_weight = 1.0 - panel_weight

    pred = (
        panel_weight * calibration["panel_ar_pred"]
        + seasonal_weight * calibration["seasonal_pred"]
    )

    mae = (
        calibration["actual_value"] - pred
    ).abs().mean()

    weight_rows.append({
        "panel_weight": panel_weight,
        "seasonal_weight": seasonal_weight,
        "MAE": mae,
    })

weight_grid = (
    pd.DataFrame(weight_rows)
    .sort_values("MAE")
    .reset_index(drop=True)
)

display(weight_grid.head(10).round(3))

best = weight_grid.iloc[0]

PANEL_WEIGHT = float(best["panel_weight"])
SEASONAL_WEIGHT = float(best["seasonal_weight"])

print(
    f"Выбранные веса: Panel={PANEL_WEIGHT:.2f}, "
    f"Seasonal={SEASONAL_WEIGHT:.2f}"
)
print("MAE на калибровке:", round(float(best["MAE"]), 3))

assert np.isclose(PANEL_WEIGHT, 0.45)
assert np.isclose(SEASONAL_WEIGHT, 0.55)

# При наличии общего YAML проверяем согласованность, не подбирая веса заново.
CONFIG_FILE = PROJECT_DIR / "configs" / "final_config.yaml"
if CONFIG_FILE.is_file():
    try:
        import yaml
    except ImportError as exc:
        raise ImportError(
            "Для чтения configs/final_config.yaml установите PyYAML."
        ) from exc
    with CONFIG_FILE.open(encoding="utf-8") as config_stream:
        final_config = yaml.safe_load(config_stream) or {}
    static_cfg = final_config.get("static_ensemble", {})
    assert np.isclose(float(static_cfg.get("panel_weight", PANEL_WEIGHT)), PANEL_WEIGHT)
    assert np.isclose(float(static_cfg.get("seasonal_weight", SEASONAL_WEIGHT)), SEASONAL_WEIGHT)
    forecast_cfg = final_config.get("forecasting", {})
    assert forecast_cfg.get("horizons", [1, 2, 3]) == [1, 2, 3]
    print("Final config: CONSISTENT")
else:
    print("Final config: not yet provided; frozen parameters retained")
assert np.isclose(float(best["MAE"]), 942.804, atol=0.15)


### Вывод по калибровке

Лучшее сочетание на раннем периоде:

- **45% Panel AR**
- **55% Seasonal**

Мы фиксируем эти веса и дальше их не меняем на основной тестовой части.

Весовая конфигурация сохранена без повторной оптимизации; ограничение по доступности июньской статистики отмечено выше.

In [ ]:
def forecast_static(origin, horizons=(1, 2, 3)):
    # Финальная статическая модель: 45% Panel AR + 55% Seasonal.
    origin = pd.Timestamp(origin)
    horizons = tuple(sorted(int(h) for h in horizons))

    panel_fc = forecast_panel_ar(
        origin=origin,
        horizons=horizons,
    )[
        [
            "series_id",
            "origin",
            "horizon",
            "target_period",
            "panel_ar_pred",
        ]
    ]

    seasonal_fc = forecast_seasonal(
        origin=origin,
        horizons=horizons,
    )[
        [
            "series_id",
            "origin",
            "horizon",
            "target_period",
            "seasonal_pred",
        ]
    ]

    result = panel_fc.merge(
        seasonal_fc,
        on=["series_id", "origin", "horizon", "target_period"],
        validate="one_to_one",
    )

    result["panel_weight"] = PANEL_WEIGHT
    result["seasonal_weight"] = SEASONAL_WEIGHT

    result["final_pred"] = (
        PANEL_WEIGHT * result["panel_ar_pred"]
        + SEASONAL_WEIGHT * result["seasonal_pred"]
    )

    assert len(result) == 2016 * len(horizons)
    assert result["final_pred"].notna().all()

    return (
        result
        .sort_values(["horizon", "series_id"])
        .reset_index(drop=True)
    )


## 5. Основная multi-horizon проверка

Теперь веса уже зафиксированы.

Оцениваем модель на четырёх последовательных origins:

- июнь 2024;
- июль 2024;
- август 2024;
- сентябрь 2024.

Для каждого origin строятся прогнозы на `1`, `2` и `3` месяца.

In [ ]:
TEST_ORIGINS = pd.to_datetime([
    "2024-06-01",
    "2024-07-01",
    "2024-08-01",
    "2024-09-01",
])

test_parts = []

actual_table = (
    panel[
        ["series_id", "period", "value"]
    ]
    .rename(
        columns={
            "period": "target_period",
            "value": "actual_value",
        }
    )
)

for origin in TEST_ORIGINS:
    pred = forecast_static(
        origin=origin,
        horizons=(1, 2, 3),
    )

    pred = pred.merge(
        actual_table,
        on=["series_id", "target_period"],
        validate="many_to_one",
    )

    test_parts.append(pred)

test_predictions = pd.concat(test_parts, ignore_index=True)

for col in ["panel_ar_pred", "seasonal_pred", "final_pred"]:
    test_predictions[f"{col}_error"] = (
        test_predictions["actual_value"]
        - test_predictions[col]
    ).abs()

assert len(test_predictions) == 24_192
assert not test_predictions.duplicated(
    ["series_id", "origin", "horizon"]
).any()

summary = pd.DataFrame([
    {
        "model": "Panel AR",
        "MAE": test_predictions["panel_ar_pred_error"].mean(),
    },
    {
        "model": "Seasonal",
        "MAE": test_predictions["seasonal_pred_error"].mean(),
    },
    {
        "model": "Static ensemble 45/55",
        "MAE": test_predictions["final_pred_error"].mean(),
    },
]).sort_values("MAE").reset_index(drop=True)

display(summary.round(3))

assert np.isclose(
    test_predictions["panel_ar_pred_error"].mean(),
    872.578,
    atol=0.15,
)
assert np.isclose(
    test_predictions["seasonal_pred_error"].mean(),
    865.874,
    atol=0.15,
)
assert np.isclose(
    test_predictions["final_pred_error"].mean(),
    790.859,
    atol=0.15,
)

print("Основная проверка: PASSED")


In [ ]:
by_horizon = (
    test_predictions
    .groupby("horizon")
    .agg(
        Panel_AR=("panel_ar_pred_error", "mean"),
        Seasonal=("seasonal_pred_error", "mean"),
        Static_Ensemble=("final_pred_error", "mean"),
    )
)

by_origin = (
    test_predictions
    .groupby("origin")
    .agg(
        Panel_AR=("panel_ar_pred_error", "mean"),
        Seasonal=("seasonal_pred_error", "mean"),
        Static_Ensemble=("final_pred_error", "mean"),
    )
)

print("По горизонту:")
display(by_horizon.round(3))

print("По origin:")
display(by_origin.round(3))

expected_h = {
    1: 691.492,
    2: 750.187,
    3: 930.897,
}

for h, expected in expected_h.items():
    assert np.isclose(
        by_horizon.loc[h, "Static_Ensemble"],
        expected,
        atol=0.15,
    )

expected_origin = {
    pd.Timestamp("2024-06-01"): 920.623,
    pd.Timestamp("2024-07-01"): 735.461,
    pd.Timestamp("2024-08-01"): 744.334,
    pd.Timestamp("2024-09-01"): 763.017,
}

for origin, expected in expected_origin.items():
    assert np.isclose(
        by_origin.loc[origin, "Static_Ensemble"],
        expected,
        atol=0.15,
    )


### Вывод

Ансамбль оказался заметно лучше каждого эксперта по отдельности:

- Panel AR: около **872.6 MAE**
- Seasonal: около **865.9 MAE**
- наш фиксированный ансамбль: **790.9 MAE**

Самый сложный горизонт — третий месяц. Поэтому дальше имеет смысл не просто искать ещё одну статическую модель, а сделать веса **адаптивными**.

## 6. Проверка устойчивости выигрыша

Проверяем, что преимущество ансамбля над двумя компонентами не держится на нескольких случайных муниципалитетах.

Пересэмплируем целые муниципальные ряды, сохраняя все их прогнозы вместе.

In [ ]:
series_gain = (
    test_predictions
    .groupby("series_id")
    .agg(
        blend_mae=("final_pred_error", "mean"),
        panel_mae=("panel_ar_pred_error", "mean"),
        seasonal_mae=("seasonal_pred_error", "mean"),
    )
)

series_gain["gain_vs_panel"] = (
    series_gain["panel_mae"] - series_gain["blend_mae"]
)

series_gain["gain_vs_seasonal"] = (
    series_gain["seasonal_mae"] - series_gain["blend_mae"]
)

rng = np.random.default_rng(42)
N_BOOT = 5000
n = len(series_gain)

def cluster_bootstrap(values):
    values = np.asarray(values, dtype=float)
    sims = np.empty(N_BOOT)

    for b in range(N_BOOT):
        idx = rng.integers(0, n, size=n)
        sims[b] = values[idx].mean()

    lo, hi = np.quantile(sims, [0.025, 0.975])

    return {
        "gain": values.mean(),
        "ci_2.5": lo,
        "ci_97.5": hi,
        "P(gain>0)": (sims > 0).mean(),
    }

bootstrap_summary = pd.DataFrame({
    "vs_panel": cluster_bootstrap(series_gain["gain_vs_panel"]),
    "vs_seasonal": cluster_bootstrap(series_gain["gain_vs_seasonal"]),
}).T

display(bootstrap_summary.round(3))


### Вывод по bootstrap

Если доверительные интервалы находятся выше нуля, значит улучшение ансамбля носит общий характер и не объясняется несколькими удачными рядами.

## 7. Более поздний forward-период

Чтобы проверить, как статическая модель переносится на следующий временной режим, используем ещё более поздние прогнозы:

- origin `2024-10-01`: `h1` и `h2`;
- origin `2024-11-01`: `h1`.

Всего получается **6 048 новых прогнозов**.

In [ ]:
forward_specs = [
    (pd.Timestamp("2024-10-01"), (1, 2)),
    (pd.Timestamp("2024-11-01"), (1,)),
]

forward_parts = []

for origin, horizons in forward_specs:
    pred = forecast_static(
        origin=origin,
        horizons=horizons,
    )

    pred = pred.merge(
        actual_table,
        on=["series_id", "target_period"],
        validate="many_to_one",
    )

    pred["abs_error"] = (
        pred["actual_value"] - pred["final_pred"]
    ).abs()

    forward_parts.append(pred)

forward_predictions = pd.concat(
    forward_parts,
    ignore_index=True,
)

print("Forward-строк:", len(forward_predictions))

forward_mae = forward_predictions["abs_error"].mean()

print("Forward MAE:", round(forward_mae, 3))

forward_cells = (
    forward_predictions
    .groupby(["origin", "horizon"])["abs_error"]
    .mean()
    .rename("MAE")
    .reset_index()
)

display(forward_cells.round(3))

assert len(forward_predictions) == 6_048
assert np.isclose(forward_mae, 858.810, atol=0.15)

expected_forward = {
    (pd.Timestamp("2024-10-01"), 1): 835.081,
    (pd.Timestamp("2024-10-01"), 2): 993.725,
    (pd.Timestamp("2024-11-01"), 1): 747.625,
}

for (origin, h), expected in expected_forward.items():
    actual = float(
        forward_cells.loc[
            (forward_cells["origin"] == origin)
            & (forward_cells["horizon"] == h),
            "MAE",
        ].iloc[0]
    )
    assert np.isclose(actual, expected, atol=0.15)

print("Forward-проверка: PASSED")


### Вывод

На более позднем периоде статическая модель даёт **MAE ≈ 858.8**.

Качество меняется вместе с временным режимом:

- прогноз на один месяц остаётся относительно устойчивым;
- двухмесячный прогноз на декабрь становится заметно сложнее.

Это главный мотив для следующего этапа: вместо постоянных весов сделать **адаптивный ансамбль**, который меняет доверие к экспертам по уже наблюдаемым ошибкам.

## 8. Сохранение артефактов

In [ ]:
TEST_FILE = INTERIM_DIR / "our_static_multihorizon_predictions.parquet"
FORWARD_FILE = INTERIM_DIR / "our_static_forward_nov_dec_predictions.parquet"
SUMMARY_FILE = INTERIM_DIR / "our_static_forecasting_summary.csv"
MANIFEST_FILE = INTERIM_DIR / "our_static_forecasting_manifest.json"

test_predictions.to_parquet(TEST_FILE, index=False)
forward_predictions.to_parquet(FORWARD_FILE, index=False)

summary_export = pd.DataFrame([
    {
        "period": "Jun-Sep multi-horizon",
        "n_forecasts": len(test_predictions),
        "MAE": test_predictions["final_pred_error"].mean(),
    },
    {
        "period": "Nov-Dec forward",
        "n_forecasts": len(forward_predictions),
        "MAE": forward_predictions["abs_error"].mean(),
    },
])

summary_export.to_csv(SUMMARY_FILE, index=False)

manifest = {
    "name": "Static Panel AR + Seasonal ensemble",
    "panel_weight": PANEL_WEIGHT,
    "seasonal_weight": SEASONAL_WEIGHT,
    "calibration_cutoff": str(CAL_CUTOFF.date()),
    "calibration_rows": len(calibration),
    "test_origins": [d.strftime("%Y-%m-%d") for d in TEST_ORIGINS],
    "test_rows": len(test_predictions),
    "test_mae": float(test_predictions["final_pred_error"].mean()),
    "forward_rows": len(forward_predictions),
    "forward_mae": float(forward_predictions["abs_error"].mean()),
    "status": "historical static forecasting benchmark; publication-lag limitation documented",
}

with open(MANIFEST_FILE, "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

artifacts = pd.DataFrame([
    {"artifact": TEST_FILE.name, "exists": TEST_FILE.exists()},
    {"artifact": FORWARD_FILE.name, "exists": FORWARD_FILE.exists()},
    {"artifact": SUMMARY_FILE.name, "exists": SUMMARY_FILE.exists()},
    {"artifact": MANIFEST_FILE.name, "exists": MANIFEST_FILE.exists()},
])

display(artifacts)
assert artifacts["exists"].all()

print("Артефакты сохранены в:", INTERIM_DIR)


## 9. Итог этапа

Мы получили собственную статическую модель из двух разных источников сигнала:

- **Panel AR** отвечает за общий рост и локальное отклонение МО;
- **Seasonal expert** отвечает за устойчивую сезонную форму.

Вес `45/55` выбран по предшествующему ретроспективному калибровочному набору (включающему целевой июнь 2024) и далее зафиксирован. Это не является доказательством доступности весов к началу июня.

Основные результаты:

- Jun–Sep multi-horizon: **MAE ≈ 790.9**
- более поздний Nov–Dec forward: **MAE ≈ 858.8**

Главный вывод простой: фиксированный ансамбль работает лучше отдельных компонентов, но качество заметно зависит от временного режима.

Следующий этап — построить **адаптивный ансамбль**, который будет автоматически менять веса экспертов по уже наблюдаемым ошибкам.